# Voxel downsampling

Load a point cloud, compute voxel keys, then replace each voxel's points by their centroid.

In [ ]:
import math
from collections.abc import Iterable

from gridsnap.snap import Point, VoxelKey, read_xyz

points = read_xyz(DEBRIEF_DATA["points"])
size = DEBRIEF_DATA["voxel_size"]
print(f"{len(points)} points, voxel size {size}")

`voxel_key` maps a point to the integer index of its voxel. Note `math.floor`: try replacing it with `int` and rerun the demo.

In [ ]:
def voxel_key(point: Point, size: float) -> VoxelKey:
    """The integer index of the voxel of edge ``size`` that contains ``point``.

    Uses floor, not int(): int() truncates toward zero, which would put -0.5
    and 0.5 in the same voxel.
    """
    if size <= 0:
        raise ValueError(f"voxel size must be positive, got {size}")
    x, y, z = point
    return (math.floor(x / size), math.floor(y / size), math.floor(z / size))

In [ ]:
for p in [(0.4, 0.2, 0.1), (-0.4, 0.2, 0.1), (1.2, -0.1, 0.0)]:
    print(p, "->", voxel_key(p, size))

`downsample` accumulates per-voxel sums and counts in a dict, then turns them into centroids.

In [ ]:
def downsample(points: Iterable[Point], size: float) -> list[Point]:
    """Replace the points in each occupied voxel by their centroid, in first-seen order."""
    sums: dict[VoxelKey, list[float]] = {}
    for x, y, z in points:
        acc = sums.setdefault(voxel_key((x, y, z), size), [0.0, 0.0, 0.0, 0.0])
        acc[0] += x
        acc[1] += y
        acc[2] += z
        acc[3] += 1
    return [(sx / n, sy / n, sz / n) for sx, sy, sz, n in sums.values()]

In [ ]:
reduced = downsample(points, size)
print(len(points), "points ->", len(reduced), "voxels")
reduced